**Google Cloud Storage** or **AWS S3** (for storing FAISS indices)
- GCS: Free tier with 5GB storage
- AWS S3: Free tier with 5GB for 12 months

In [3]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import numpy as np
import pandas as pd
from common.const import SEMANTIC_VECTOR_DB_COLLECTION, DEVICE, SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS
from common.helper import clickhouse_extract_data_as_df, connect_qdrant
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
import faiss


# 1. Query data

In [4]:
from common.const import SCHEMA, TABLE_RAW_PRODUCTS, DEVICE
from common.helper import clickhouse_extract_data_as_df

# Products
products_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    main_category,
    details,
    CAST(price AS Nullable(Float64)) AS price,
    average_rating,
    rating_number
FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
LIMIT 10000
"""
products_df = clickhouse_extract_data_as_df(products_query)

# Reviews
reviews_query = f"""
SELECT 
    user_id,
    parent_asin,
    title,
    text,
    rating
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
LIMIT 10000
"""
reviews_df = clickhouse_extract_data_as_df(reviews_query)

In [5]:
def create_product_text(row):
    text_parts = []

    if pd.notna(row['title']):
        text_parts.append((row['title'] + " ") * 1 + row['title']) # duplicate once

    if pd.notna(row['description']):
        text_parts.append(row['description'])

    if pd.notna(row['features']):
        text_parts.append(row['features'])

    if pd.notna(row['main_category']):
        text_parts.append((row['main_category'] + " ") * 2 + row['main_category'])
    
    if pd.notna(row['details']):
        text_parts.append(row['details'])

    return ' '.join(text_parts)

products_df['combined_text'] = products_df.apply(create_product_text, axis=1)

In [ ]:
reviews_df['full_review'] = (
    reviews_df['title'].fillna('') + ' ' +
    reviews_df['text'].fillna('')
)

user_reviews = (
    reviews_df
    .groupby('user_id')['full_review']
    .apply(' '.join)
    .reset_index(name='all_reviews_text')
)

# 2. Generate similarity computation for similarity product

## 2.1. SentenceTransformer

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings (384 dimensions)
semantic_embeddings = model.encode(
    products_df['combined_text'].tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,  # Normalize for cosine similarity
    device=DEVICE
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [8]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings (384 dimensions)
print("Encoding products (this may take a few minutes)...")
user_embeddings = model.encode(
    user_reviews['all_reviews_text'].tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,  # Normalize for cosine similarity
    device=DEVICE
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding products (this may take a few minutes)...


Batches:   0%|          | 0/306 [00:00<?, ?it/s]

## 2.2. Insert vectors

In [22]:
import faiss
import numpy as np

dim = 384

# Normalize once BEFORE inserting
faiss.normalize_L2(semantic_embeddings)

# Create index once
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))

BATCH_SIZE = 500

for i in range(0, len(semantic_embeddings), BATCH_SIZE):
    batch_vectors = semantic_embeddings[i:i+BATCH_SIZE].astype("float32")

    # Use dataframe index as semantic ID
    batch_ids = products_df.index[i:i+len(batch_vectors)].to_numpy().astype("int64")

    index.add_with_ids(batch_vectors, batch_ids)

    # Save after each batch
    faiss.write_index(index, "semantic.index")

    print(f"Inserted {i + len(batch_vectors)} / {len(semantic_embeddings)}")

Inserted 500 / 10000
Inserted 1000 / 10000
Inserted 1500 / 10000
Inserted 2000 / 10000
Inserted 2500 / 10000
Inserted 3000 / 10000
Inserted 3500 / 10000
Inserted 4000 / 10000
Inserted 4500 / 10000
Inserted 5000 / 10000
Inserted 5500 / 10000
Inserted 6000 / 10000
Inserted 6500 / 10000
Inserted 7000 / 10000
Inserted 7500 / 10000
Inserted 8000 / 10000
Inserted 8500 / 10000
Inserted 9000 / 10000
Inserted 9500 / 10000
Inserted 10000 / 10000


## 2.3. Query similar products

In [ ]:
asin_to_index = dict(
    zip(products_df["parent_asin"], products_df.index)
) #to get fast index from asin for querying (o1 lookup)

def get_similar_products_full(product_asin: str, top_k: int = 10):
    index = faiss.read_index("semantic.index")
    if product_asin not in asin_to_index:
        raise ValueError("ASIN not found")

    # Step 1: Convert ASIN → numeric index
    product_id = asin_to_index[product_asin]

    # Step 2: Get query vector
    query_vector = semantic_embeddings[product_id].reshape(1, -1).astype("float32")

    # Step 3: Search FAISS (get one extra for self match)
    D, I = index.search(query_vector, top_k + 1)

    results = []

    for rank, idx in enumerate(I[0]):
        if idx == product_id:
            continue  # skip itself

        product_details = products_df.loc[idx].to_dict()
        vector = semantic_embeddings[idx]

        results.append({
            "rank": rank,
            "similar_product_index": int(idx),
            "similar_product_asin": product_details["parent_asin"],
            "similarity_score": float(D[0][rank]),
            "product_details": product_details,
            "vector": vector
        })

        if len(results) == top_k:
            break

    return results

results = get_similar_products_full("0005946468", top_k=10)

for r in results:
    print("Rank:", r["rank"])
    print("ASIN:", r["similar_product_asin"])
    print("Score:", r["similarity_score"])
    print("Title:", r["product_details"]["title"])
    # print("Vector shape:", r["vector"].shape)
    print("-" * 50)


Rank: 1
ASIN: B004YT8D5W
Score: 0.7118107080459595
Title: Lanolin, Manuka Honey and Collagen Facial Moisturizer
--------------------------------------------------
Rank: 2
ASIN: B007L4LVP6
Score: 0.7030024528503418
Title: Platone Gift Pack ~ Moisturizing Face Creme 2.5oz. ~ Rejuvenating Eye Creme 1oz.
--------------------------------------------------
Rank: 3
ASIN: B0090007DQ
Score: 0.6847703456878662
Title: 2 Bar Set Venezia Soapworks Oatmeal Moisturizing Soap
--------------------------------------------------
Rank: 4
ASIN: B0065JPB36
Score: 0.6831436157226562
Title: Curel Ultra Healing Lotion 2.50 oz (Pack of 2)
--------------------------------------------------
Rank: 5
ASIN: B00GMOZQX4
Score: 0.6803656816482544
Title: Kiss My Face Moisturizing Hand Soap, Anjou Pear 9 oz ( Pack of 3)
--------------------------------------------------
Rank: 6
ASIN: B00HT6AH4M
Score: 0.6722538471221924
Title: Revlon PhotoReady Cream Blush, Flushed [200] 0.44 oz (Pack of 2)
------------------------------

# 3. Generate similarity computation for similar users

## 3.1. Feature engineering

In [27]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings (384 dimensions)
print("Encoding products (this may take a few minutes)...")
user_embeddings = model.encode(
    user_reviews['all_reviews_text'].tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,  # Normalize for cosine similarity
    device=DEVICE
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding products (this may take a few minutes)...


Batches:   0%|          | 0/306 [00:00<?, ?it/s]

## 3.2. Insert Vectors

In [34]:
import faiss
import numpy as np

dim = 384

# Normalize once BEFORE inserting
faiss.normalize_L2(user_embeddings)

# Create index once
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))

BATCH_SIZE = 500

for i in range(0, len(user_embeddings), BATCH_SIZE):
    batch_vectors = user_embeddings[i:i+BATCH_SIZE].astype("float32")

    # Use dataframe index as semantic ID
    batch_ids = user_reviews.index[i:i+len(batch_vectors)].to_numpy().astype("int64")

    index.add_with_ids(batch_vectors, batch_ids)

    # Save after each batch
    faiss.write_index(index, "user_semantic.index")

    print(f"Inserted {i + len(batch_vectors)} / {len(user_embeddings)}")

Inserted 500 / 9772
Inserted 1000 / 9772
Inserted 1500 / 9772
Inserted 2000 / 9772
Inserted 2500 / 9772
Inserted 3000 / 9772
Inserted 3500 / 9772
Inserted 4000 / 9772
Inserted 4500 / 9772
Inserted 5000 / 9772
Inserted 5500 / 9772
Inserted 6000 / 9772
Inserted 6500 / 9772
Inserted 7000 / 9772
Inserted 7500 / 9772
Inserted 8000 / 9772
Inserted 8500 / 9772
Inserted 9000 / 9772
Inserted 9500 / 9772
Inserted 9772 / 9772


## 3.3. Query similar User

In [37]:
id_to_index = dict(
    zip(user_reviews["user_id"], user_reviews.index)
) #to get fast index from asin for querying (o1 lookup)

def get_similar_users_full(user_id: str, top_k: int = 10):
    #cuz user_id have alphanumeric, need map to numeric index for faiss, user_id: "A1B2C3D4E5", user_index: 12345, index is faiss
    index = faiss.read_index("user_semantic.index")
    if user_id not in id_to_index:
        raise ValueError("User ID not found")

    # Step 1: Convert ASIN → numeric index
    user_index = id_to_index[user_id]

    # Step 2: Get query vector
    query_vector = user_embeddings[user_index].reshape(1, -1).astype("float32")

    # Step 3: Search FAISS (get one extra for self match)
    D, I = index.search(query_vector, top_k + 1)

    results = []

    for rank, idx in enumerate(I[0]):
        if idx == user_index:
            continue  # skip itself

        user_details = user_reviews.loc[idx].to_dict()
        vector = user_embeddings[idx]

        results.append({
            "rank": rank,
            "similar_user_index": int(idx),
            "similar_user_id": user_details["user_id"],
            "similarity_score": float(D[0][rank]),
            "user_all_review": user_details['all_reviews_text'],
            "vector": vector
        })

        if len(results) == top_k:
            break

    return results

results = get_similar_users_full("AE222Y4WTST6BUZ4J5Y2H6QMBITQ", top_k=10)

for r in results:
    print("Rank:", r["rank"])
    print("User ID:", r["similar_user_id"])
    print("Score:", r["similarity_score"])
    print("All Reviews:", r["user_all_review"][:100])  # Show first 100 characters of reviews
    # print("Vector shape:", r["vector"].shape)
    print("-" * 50)


Rank: 1
User ID: AHLRNSZNP52TPQLVY63YY2YGC32A
Score: 0.6488837003707886
All Reviews: Perfect for contact lens wearers Last year I bought new contact lenses, and they quickly got scratch
--------------------------------------------------
Rank: 2
User ID: AGCNPIF4WYAFLMVEG3G26XWFSFSQ
Score: 0.6449755430221558
All Reviews: Great hand cleaner! I use this to make sure my hands are clean before putting my contact lenses in. 
--------------------------------------------------
Rank: 3
User ID: AHRTADTNDEDMM2WIR7YYYWLJ43YQ
Score: 0.5583561062812805
All Reviews: Must Have for Contact Lens Users It's extremely hard to find soap these days without moisturizers or
--------------------------------------------------
Rank: 4
User ID: AEZMCSXDKHHOFJGOMABGHSAGELGA
Score: 0.5565992593765259
All Reviews: Perfect for contact wearers My eye dr recommended this soap to me years ago after I had some issues 
--------------------------------------------------
Rank: 5
User ID: AFV3BXPYO44GSPP57MNWY4D5OHDA
Score: